In [2]:
import pandas as pd

# Define the file path
file_path = "../raw_data/Sales_Satisfaction_with_NaNs.csv"

# Load the dataset
df = pd.read_csv(file_path)

# Display shape, first 5 rows and null values
print("Dataset Shape:", df.shape)
display(df.head())
print('Missing values per column: ')
print(df.isnull().sum())

Dataset Shape: (10000, 11)


,Group,Customer_Segment,Sales_Before,Sales_After,Customer_Satisfaction_Before,Customer_Satisfaction_After,Purchase_Made,Sales_Growth,Satisfaction_Growth,Engagement_Score,ROI_index
0,Control,High Value,240.548359,300.007567,74.684767,NaN,No,59.459209,-74.684767,18.52434,24.718193
1,Treatment,High Value,246.862114,381.337555,100.000000,100.000000,Yes,134.475440,0.000000,24.60000,54.473908
2,Control,High Value,156.978084,179.330464,98.780735,100.000000,No,22.352379,1.219265,24.30738,14.239172
3,Control,Medium Value,192.126708,229.278031,49.333766,39.811841,Yes,37.151323,-9.521925,12.44010,19.336886
4,NaN,High Value,229.685622,NaN,83.974852,87.738591,Yes,-229.685622,3.763739,20.75396,-100.000000


Missing values per column: 
Group                           1401
Customer_Segment                1966
Sales_Before                    1522
Sales_After                      767
Customer_Satisfaction_Before    1670
Customer_Satisfaction_After     1640
Purchase_Made                    805
Sales_Growth                       0
Satisfaction_Growth                0
Engagement_Score                   0
ROI_index                          0
dtype: int64


In [3]:
# Fill missing values in categorical columns with 'Unknown'
df["Customer_Segment"] = df["Customer_Segment"].fillna("Unknown")
df["Group"] = df["Group"].fillna("Unknown")

# Check if it worked for these two columns
print("Missing values after first fix:")
print(df[["Customer_Segment", "Group"]].isnull().sum())

Missing values after first fix:
Customer_Segment    0
Group               0
dtype: int64


In [4]:
# First, fill missing values in numerical columns using the median of each segment
num_cols = [
    "Sales_Before",
    "Sales_After",
    "Customer_Satisfaction_Before",
    "Customer_Satisfaction_After",
]

for col in num_cols:
    df[col] = df.groupby("Customer_Segment")[col].transform(
                lambda x: x.fillna(x.median())
    )

# Now that Sales_After has no NaNs, re-run the conditional fix for the remaining 61 rows
df.loc[(df["Purchase_Made"].isnull()) & (df["Sales_After"] == 0), "Purchase_Made"] = (
    "No"
)
print("Missing values after first fix:")
print(df[["Sales_Before",
    "Sales_After",
    "Customer_Satisfaction_Before",
    "Customer_Satisfaction_After",]].isnull().sum())

Missing values after first fix:
Sales_Before                    0
Sales_After                     0
Customer_Satisfaction_Before    0
Customer_Satisfaction_After     0
dtype: int64


In [5]:
# If Sales_After > 0 and Purchase_Made is NaN -> Fill with 'Yes'
df.loc[(df["Purchase_Made"].isnull()) & (df["Sales_After"] > 0), "Purchase_Made"] = "Yes"

# If Sales_After == 0 and Purchase_Made is NaN -> Fill with 'No'
df.loc[(df["Purchase_Made"].isnull()) & (df["Sales_After"] == 0), "Purchase_Made"] = "No"

# Check if there are any NaNs left in Purchase_Made
print("Missing values in Purchase_Made:")
print(df["Purchase_Made"].isnull().sum())

Missing values in Purchase_Made:
0


In [6]:
# Calculate absolute sales delta
df["Sales_Delta"] = df["Sales_After"] - df["Sales_Before"]

# Calculate percentage growth - efficiency
# We add a tiny 0.001 to avoid Division by Zero errors if Sales_Before is 0
df["Sales_Efficiency"] = (df["Sales_After"] - df["Sales_Before"]) / (df["Sales_Before"] + 0.001)

# Calculate satisfaction change
df["Satisfaction_Delta"] = df["Customer_Satisfaction_After"] - df["Customer_Satisfaction_Before"]

display(df[["Customer_Segment", "Sales_Delta", "Sales_Efficiency", "Satisfaction_Delta"]].head())

,Customer_Segment,Sales_Delta,Sales_Efficiency,Satisfaction_Delta
0,High Value,59.459209,0.247181,19.923499
1,High Value,134.475440,0.544737,0.000000
2,High Value,22.352379,0.142391,1.219265
3,Medium Value,37.151323,0.193368,-9.521925
4,High Value,69.146685,0.301048,3.763739


In [ ]:
# saving new dataset in MySQL database for queries

from sqlalchemy import create_engine
import pymysql

# Parameters
USER = "root"
PASSWORD = "INSERT MYSQL PASSWORD"  
HOST = "localhost"
PORT = "3306"  
DATABASE = "marketing_analytics"

try:
    # Create connection engine
    connection_string = f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}"
    engine = create_engine(connection_string)

   # Stream the clean DataFrame directly into a MySQL table
    df.to_sql(name="marketing_campaign_data", con=engine, if_exists="replace", index=False)

    print("Success")

except Exception as e:
    print(f"Error")

Success
